# **YOLOv8 Training**
**YOLOv8 Training & Experiments Notebook**\
**Project:** SEEZY Supermarket Product Detection\
**Target:** Google Colab GPU Environment\
**Authors:** Ben Keinan & Yovel Ben Hamo

## **Experiment A**
**Model Scale Comparison (v8n vs v8s vs v8m)**


### Instructions

**How to Choose the Winner in Experiment A?**\
The goal here is to find the optimal balance between detection accuracy and hardware performance for your specific edge device.
* Do not rely solely on accuracy: You cannot pick the winner just by looking at the highest mAP@0.5 or mAP@0.5:0.95 from the Google Colab training. A larger model like YOLOv8m will almost always have a higher mAP in a cloud environment, but that does not mean it will run well on a mobile robot.

* Evaluate Jetson Benchmarks: The true winner is determined by how the models perform on the NVIDIA Jetson Orin Nano. You must look at the inference speed (latency in ms) and the resulting Frames Per Second (FPS).

* Require GPU Headroom: The robot must run other demanding ROS 2 nodes simultaneously (such as SLAM, LiDAR processing, and ESP32 communication). If a model consumes 100% of the Jetson's GPU to achieve its FPS, or if it runs so hot that it causes thermal throttling, it must be rejected. You choose the model that gives the highest mAP while still leaving enough spare computational headroom to maintain a smooth, real-time pipeline.

### SECTION 1: Setup Environment

In [ ]:
# ==========================================
# SECTION 1: Setup Environment
# ==========================================
# Install dependencies quietly
!pip install ultralytics pandas matplotlib seaborn --quiet

import os
import shutil
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

# Safety verification for GPU acceleration
assert torch.cuda.is_available(), "❌ GPU is NOT enabled! Go to Runtime → Change runtime type → GPU"
print("✅ Hardware Accelerator active:", torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 5.6 MB/s eta 0:00:00


KeyboardInterrupt: 

### SECTION 2: Google Drive Setup & Workspace

In [ ]:
# ==========================================
# SECTION 2: Google Drive Setup & Workspace
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

BASE_PROJECT_PATH = "Enter Path"
DATASET_PATH = f"{BASE_PROJECT_PATH}/dataset"
EXPERIMENTS_BACKUP_PATH = f"{BASE_PROJECT_PATH}/experiments_backup"

os.makedirs(EXPERIMENTS_BACKUP_PATH, exist_ok=True)

print("✅ Persistent Drive root:", BASE_PROJECT_PATH)
print("✅ Experiment outputs directory:", EXPERIMENTS_BACKUP_PATH)

### SECTION 3: Dataset Configuration & Verification

In [ ]:
# ==========================================
# SECTION 3: Dataset Configuration & Verification
# ==========================================
yaml_content = f"""
path: {DATASET_PATH}

train: images/train
val: images/val

names:
  0: milk_3p_tnuva
  1: ketchup_heinz
  2: bamba_osem
  3: cafe_names_elit
  4: toothpaste_colgate
"""

yaml_path = f"{DATASET_PATH}/data.yaml"

with open(yaml_path, "w") as f:
    f.write(yaml_content)

print("✅ data.yaml configured at:", yaml_path)

# Verify image & label counts to prevent silent path errors
train_imgs = len(os.listdir(f"{DATASET_PATH}/images/train")) if os.path.exists(f"{DATASET_PATH}/images/train") else 0
val_imgs = len(os.listdir(f"{DATASET_PATH}/images/val")) if os.path.exists(f"{DATASET_PATH}/images/val") else 0

print(f"📊 Current Dataset Distribution: Train = {train_imgs} images | Val = {val_imgs} images")

### SECTION 4: Experiment Execution

In [ ]:
# ==========================================
# SECTION 4: Experiment Execution
# ==========================================
# Baseline hyperparameters shared across models to isolate the architectural variable
BASE_PARAMS = {
    'data': yaml_path,
    'epochs': 50,
    'imgsz': 640,
    'batch': 16,
    'device': 0,
    'patience': 15,
    'project': EXPERIMENTS_BACKUP_PATH,
}

# --- Experiment A.1: YOLOv8 Nano (Lightweight Baseline) ---
print("\n🚀 Starting Experiment A.1: YOLOv8n (Nano Model)...")
model_nano = YOLO("yolov8n.pt")
results_nano = model_nano.train(
    name='expA_yolov8n',
    exist_ok=True,
    **BASE_PARAMS
)

# --- Experiment A.2: YOLOv8 Small (Balanced Model) ---
print("\n🚀 Starting Experiment A.2: YOLOv8s (Small Model)...")
model_small = YOLO("yolov8s.pt")
results_small = model_small.train(
    name='expA_yolov8s',
    exist_ok=True,
    **BASE_PARAMS
)

# --- Experiment A.3: YOLOv8 Medium (Heavyweight Capacity) ---
print("\n🚀 Starting Experiment A.3: YOLOv8m (Medium Model)...")
model_medium = YOLO("yolov8m.pt")
results_medium = model_medium.train(
    name='expA_yolov8m',
    exist_ok=True,
    **BASE_PARAMS
)

### SECTION 5: Metric Harvester & Experiment Comparison

In [ ]:
# ==========================================
# SECTION 5: Metric Harvester & Experiment Comparison
# ==========================================
def parse_experiment_results(run_name):
    csv_path = f"{EXPERIMENTS_BACKUP_PATH}/{run_name}/results.csv"
    if not os.path.exists(csv_path):
        print(f"⚠️ Results file not found at: {csv_path}")
        return None

    try:
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip() # Clean whitespace from headers
        best_epoch = df.iloc[-1]            # Extract metrics from the final best epoch
        return {
            'Experiment': run_name,
            'Epochs Trained': len(df),
            'Precision': round(best_epoch['metrics/precision(B)'], 4),
            'Recall': round(best_epoch['metrics/recall(B)'], 4),
            'mAP@0.5': round(best_epoch['metrics/mAP50(B)'], 4),
            'mAP@0.5:0.95': round(best_epoch['metrics/mAP50-95(B)'], 4),
        }
    except Exception as e:
        print(f"⚠️ Error parsing results for {run_name}: {e}")
        return None

# Harvest metrics from all three runs
experiments = ['expA_yolov8n', 'expA_yolov8s', 'expA_yolov8m']
summary_data = [data for exp_name in experiments if (data := parse_experiment_results(exp_name)) is not None]

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    print("\n=======================================================")
    print("          EXPERIMENT A: MODEL SCALE COMPARISON         ")
    print("=======================================================")
    print(summary_df.to_string(index=False))

    summary_csv_path = f"{EXPERIMENTS_BACKUP_PATH}/expA_model_scale_summary.csv"
    summary_df.to_csv(summary_csv_path, index=False)
    print(f"\n📊 Summary table saved to Drive: {summary_csv_path}")
else:
    print("❌ No experiment results could be harvested.")

### SECTION 6: Deployment Export (All 3 Models to ONNX)

In [ ]:
# ==========================================
# SECTION 6: Deployment Export (All 3 Models to ONNX)
# ==========================================
# Export ALL THREE trained models to ONNX so they can be benchmarked on the Jetson
onnx_output_dir = f"{BASE_PROJECT_PATH}/onnx_models"
os.makedirs(onnx_output_dir, exist_ok=True)

successful_exports = []
failed_exports = []

for exp_name in experiments:
    pt_path = f"{EXPERIMENTS_BACKUP_PATH}/{exp_name}/weights/best.pt"

    print(f"\n📦 Exporting {exp_name} to ONNX format...")
    try:
        model_to_export = YOLO(pt_path)
        onnx_file = model_to_export.export(format="onnx", imgsz=640)    # Export model to ONNX

        # Copy ONNX file to persistent Google Drive folder
        destination_onnx = f"{onnx_output_dir}/{exp_name}.onnx"
        shutil.copy(onnx_file, destination_onnx)
        print(f"  ✅ Exported successfully: {destination_onnx}")
        successful_exports.append(exp_name)
    except Exception as e:
        print(f"  ❌ Failed to export {exp_name}: {e}")
        failed_exports.append(exp_name)

# Accurate dynamic status report
print("\n=======================================================")
print(f"Export Status: {len(successful_exports)}/{len(experiments)} Succeeded")
if failed_exports:
    print(f"⚠️ Failed models: {failed_exports}")
if len(successful_exports) == len(experiments):
    print("🎉 All ONNX models exported and ready for Jetson benchmarking!")
print("=======================================================")

### SECTION 7: Shut Down Virtual Machine

In [ ]:
from google.colab import runtime
runtime.unassign()

## **Experiment B**
**Training Duration & Convergence (50 vs. 100 vs. 150 Epochs)**

### Instructions

**How to Choose the Winner in Experiment B?**\
You evaluate and pick the winning epoch count strictly using the Colab data curves and output table:
* Convergence / Plateau: Look at the $\text{mAP@0.5}$ progression. If $100$
epochs improves over $50$, but $150$ epochs produces no meaningful gain, training beyond 100 is unnecessary.*
* Overfitting Check: Look at the loss curves (results.png). If training loss continues dropping while validation loss starts rising, the model has begun overfitting.

### SECTION 1: Setup Environment & Dependencies

In [ ]:
# ==========================================
# SECTION 1: Setup Environment & Dependencies
# ==========================================
!pip install ultralytics pandas matplotlib seaborn --quiet

import os
import shutil
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

# Hardware accelerator check
assert torch.cuda.is_available(), "❌ GPU is NOT enabled! Go to Runtime → Change runtime type → GPU"
print(f"✅ Hardware Accelerator Active: {torch.cuda.get_device_name(0)}")

### SECTION 2: Google Drive Mounting & Paths

In [ ]:
# ==========================================
# SECTION 2: Google Drive Mounting & Paths
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

BASE_PROJECT_PATH = "Enter Path"
DATASET_PATH = f"{BASE_PROJECT_PATH}/dataset"
EXPERIMENTS_BACKUP_PATH = f"{BASE_PROJECT_PATH}/experiments_backup"
ONNX_OUTPUT_DIR = f"{BASE_PROJECT_PATH}/onnx_models"
YAML_PATH = f"{DATASET_PATH}/data.yaml"

os.makedirs(EXPERIMENTS_BACKUP_PATH, exist_ok=True)
os.makedirs(ONNX_OUTPUT_DIR, exist_ok=True)

assert os.path.exists(YAML_PATH), f"❌ data.yaml not found at: {YAML_PATH}. Verify your dataset directory!"
print(f"✅ Dataset data.yaml verified at: {YAML_PATH}")

### SECTION 3: Experiment Configuration

In [ ]:
# ==========================================
# SECTION 3: Experiment Configuration
# ==========================================
# NOTE: Set this to the winning architecture identified from Experiment A:
# e.g., "yolov8s.pt" or "yolov8n.pt"
CHOSEN_MODEL_WEIGHTS = "yolov8n.pt"

# List of epoch runs to test
EPOCH_TARGETS = [50, 100, 150]

BASE_PARAMS = {
    'data': YAML_PATH,
    'imgsz': 640,
    'batch': 16,
    'device': 0,
    'project': EXPERIMENTS_BACKUP_PATH,  # Saves directly to Drive in real-time
    'patience': 0,  # Disable early stopping so we can observe the full loss curves up to the target epoch
}

print(f"🎯 Target Architecture for Experiment B: {CHOSEN_MODEL_WEIGHTS}")
print(f"🎯 Epoch Counts to Evaluate: {EPOCH_TARGETS}")

### SECTION 4: Training Execution (50 vs 100 vs 150 Epochs)

In [ ]:
# ==========================================
# SECTION 4: Training Execution (50 vs 100 vs 150 Epochs)
# ==========================================
#for epochs in EPOCH_TARGETS:
    #exp_name = f"expB_{os.path.splitext(CHOSEN_MODEL_WEIGHTS)[0]}_{epochs}e"
exp_name = f"expB_{os.path.splitext(CHOSEN_MODEL_WEIGHTS)[0]}_150e"
print(f"\n=======================================================")
#print(f"🚀 Starting Experiment B: {exp_name} ({epochs} Epochs)...")
print(f"🚀 Starting Experiment B: {exp_name} (150 Epochs)...")
print(f"=======================================================")

model = YOLO(CHOSEN_MODEL_WEIGHTS)
model.train(
    name=exp_name,
    #epochs=epochs,
    epochs=150,
    exist_ok=True,
    **BASE_PARAMS
)
print(f"✅ Finished run: {exp_name}")

### SECTION 5: Metric Extraction & Summary Table

In [ ]:
# ==========================================
# SECTION 5: Metric Extraction & Summary Table
# ==========================================
def parse_experiment_b_results(epochs, base_model_name):
    exp_name = f"expB_{base_model_name}_{epochs}e"
    csv_path = f"{EXPERIMENTS_BACKUP_PATH}/{exp_name}/results.csv"

    if not os.path.exists(csv_path):
        print(f"⚠️ Warning: results.csv not found for {exp_name}")
        return None

    try:
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()

        # Best epoch based on highest mAP@0.5
        best_idx = df['metrics/mAP50(B)'].idxmax()
        best_row = df.loc[best_idx]
        last_row = df.iloc[-1]

        return {
            'Experiment': exp_name,
            'Configured Epochs': epochs,
            'Best Epoch': int(best_row['epoch']),
            'Precision': round(best_row['metrics/precision(B)'], 4),
            'Recall': round(best_row['metrics/recall(B)'], 4),
            'mAP@0.5 (Best)': round(best_row['metrics/mAP50(B)'], 4),
            'mAP@0.5:0.95 (Best)': round(best_row['metrics/mAP50-95(B)'], 4),
            'Final Val Box Loss': round(last_row['val/box_loss'], 4),
            'Final Val Cls Loss': round(last_row['val/cls_loss'], 4)
        }
    except Exception as e:
        print(f"⚠️ Error parsing results for {exp_name}: {e}")
        return None

base_model_tag = os.path.splitext(CHOSEN_MODEL_WEIGHTS)[0]
summary_data = [
    data for epochs in EPOCH_TARGETS
    if (data := parse_experiment_b_results(epochs, base_model_tag)) is not None
]

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    print("\n" + "=" * 70)
    print("        EXPERIMENT B: TRAINING DURATION & EPOCH CONVERGENCE     ")
    print("=" * 70)
    print(summary_df.to_string(index=False))

    summary_csv_path = f"{EXPERIMENTS_BACKUP_PATH}/expB_epoch_comparison_summary.csv"
    summary_df.to_csv(summary_csv_path, index=False)
    print(f"\n📊 Summary table exported to: {summary_csv_path}")

### SECTION 6: Overfitting & Convergence Visualizer

In [ ]:
# ==========================================
# SECTION 6: Overfitting & Convergence Visualizer
# ==========================================
# Plots Train Loss vs Validation Loss and mAP curves across epochs for the 150-epoch run
longest_run_name = f"expB_{base_model_tag}_150e"
csv_path_150 = f"{EXPERIMENTS_BACKUP_PATH}/{longest_run_name}/results.csv"

if os.path.exists(csv_path_150):
    df_150 = pd.read_csv(csv_path_150)
    df_150.columns = df_150.columns.str.strip()

    plt.figure(figsize=(14, 5))

    # Subplot 1: Loss Curves (Checking for Overfitting)
    plt.subplot(1, 2, 1)
    plt.plot(df_150['epoch'], df_150['train/box_loss'], label='Train Box Loss', color='blue')
    plt.plot(df_150['epoch'], df_150['val/box_loss'], label='Val Box Loss', color='orange', linestyle='--')
    plt.plot(df_150['epoch'], df_150['train/cls_loss'], label='Train Cls Loss', color='green')
    plt.plot(df_150['epoch'], df_150['val/cls_loss'], label='Val Cls Loss', color='red', linestyle='--')
    plt.title("Loss Convergence & Overfitting Check (150 Epochs)")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)

    # Subplot 2: mAP Trajectory
    plt.subplot(1, 2, 2)
    plt.plot(df_150['epoch'], df_150['metrics/mAP50(B)'], label='mAP@0.5', color='purple', linewidth=2)
    plt.plot(df_150['epoch'], df_150['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', color='teal', linewidth=2)
    plt.title("mAP Progression Across Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.legend()
    plt.grid(True)

    plot_save_path = f"{EXPERIMENTS_BACKUP_PATH}/expB_convergence_curves.png"
    plt.tight_layout()
    plt.savefig(plot_save_path, dpi=300)
    plt.show()
    print(f"📈 Convergence plot saved to: {plot_save_path}")

### SECTION 7: Runtime Disconnect (Safety)

In [ ]:
# ==========================================
# SECTION 7: Runtime Disconnect (Safety)
# ==========================================
from google.colab import runtime
runtime.unassign()

## **Experiment C**
**Input Resolution (640x640 vs. 832x832)**

### Instructions

**How to Choose the Winner in Experiment C?**\
This experiment tests whether feeding larger, higher-resolution images into the model yields better results for your specific supermarket environment.

* Check for Small Object Improvements: You only consider switching to 832x832 if the training data shows a significant improvement in detecting small, partially occluded, or distant products on the shelves.

* Weigh the Computational Cost: Processing 832x832 images requires significantly more computational resources than the 640x640 baseline. This will directly increase inference latency and lower your FPS on the Jetson Orin Nano.

* The Decision Rule: You should only declare 832x832 the winner if the gain in accuracy (especially Recall) is substantial and the Jetson can still comfortably process the frames at your required real-time speed. If the accuracy improvement is minor, or if the higher resolution throttles the robot's hardware, the 640x640 baseline remains the winner.

### SECTION 1: Setup Environment & Dependencies

In [ ]:
# ==========================================
# SECTION 1: Setup Environment & Dependencies
# ==========================================
!pip install ultralytics pandas matplotlib seaborn --quiet

import os
import shutil
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

# Hardware accelerator check
assert torch.cuda.is_available(), "❌ GPU is NOT enabled! Go to Runtime → Change runtime type → GPU"
print(f"✅ Hardware Accelerator Active: {torch.cuda.get_device_name(0)}")

### SECTION 2: Google Drive Mounting & Paths

In [ ]:
# ==========================================
# SECTION 2: Google Drive Mounting & Paths
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

BASE_PROJECT_PATH = "Enter Path"
DATASET_PATH = f"{BASE_PROJECT_PATH}/dataset"
EXPERIMENTS_BACKUP_PATH = f"{BASE_PROJECT_PATH}/experiments_backup"
ONNX_OUTPUT_DIR = f"{BASE_PROJECT_PATH}/onnx_models"
YAML_PATH = f"{DATASET_PATH}/data.yaml"

os.makedirs(EXPERIMENTS_BACKUP_PATH, exist_ok=True)
os.makedirs(ONNX_OUTPUT_DIR, exist_ok=True)

assert os.path.exists(YAML_PATH), f"❌ data.yaml not found at: {YAML_PATH}. Verify your dataset directory!"
print(f"✅ Dataset data.yaml verified at: {YAML_PATH}")

### SECTION 3: Experiment Configuration

In [ ]:
# ==========================================
# SECTION 3: Experiment Configuration
# ==========================================
# NOTE: Set to the winning architecture identified from Experiment A
CHOSEN_MODEL_WEIGHTS = "yolov8n.pt"

# NOTE: Set to the winning epoch count identified from Experiment B (e.g., 50, 100)
CHOSEN_EPOCHS = 100

# Resolutions to evaluate
RESOLUTIONS_TO_TEST = [640, 832]

BASE_PARAMS = {
    'data': YAML_PATH,
    'epochs': CHOSEN_EPOCHS,
    'batch': 16,        # Lower to 8 if 832x832 triggers GPU Out-Of-Memory
    'device': 0,
    'patience': 15,
    'project': EXPERIMENTS_BACKUP_PATH,  # Saves directly to Drive in real-time
}

print(f"🎯 Target Architecture: {CHOSEN_MODEL_WEIGHTS} at {CHOSEN_EPOCHS} Epochs")
print(f"🎯 Image Resolutions to Test: {RESOLUTIONS_TO_TEST}")

### SECTION 4: Training Execution (640 vs 832)

In [ ]:
# ==========================================
# SECTION 4: Training Execution (640 vs 832)
# ==========================================
#for imgsz in RESOLUTIONS_TO_TEST:
#exp_name = f"expC_{os.path.splitext(CHOSEN_MODEL_WEIGHTS)[0]}_res{imgsz}"
exp_name = f"expC_{os.path.splitext(CHOSEN_MODEL_WEIGHTS)[0]}_res832"
print(f"\n=======================================================")
print(f"🚀 Starting Experiment C: {exp_name} (Resolution: 832x832)...")
print(f"=======================================================")

model = YOLO(CHOSEN_MODEL_WEIGHTS)
model.train(
    name=exp_name,
    imgsz=832,
    exist_ok=True,
    **BASE_PARAMS
)
print(f"✅ Finished run: {exp_name}")

### SECTION 5: Metric Extraction & Summary Table

In [ ]:
# ==========================================
# SECTION 5: Metric Extraction & Summary Table
# ==========================================
def parse_experiment_c_results(imgsz, base_model_name):
    exp_name = f"expC_{base_model_name}_res{imgsz}"
    csv_path = f"{EXPERIMENTS_BACKUP_PATH}/{exp_name}/results.csv"

    if not os.path.exists(csv_path):
        print(f"⚠️ Warning: results.csv not found for {exp_name}")
        return None

    try:
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()

        # Best epoch based on highest mAP@0.5
        best_idx = df['metrics/mAP50(B)'].idxmax()
        best_row = df.loc[best_idx]

        return {
            'Experiment': exp_name,
            'Resolution': f"{imgsz}x{imgsz}",
            'Epochs Trained': len(df),
            'Precision': round(best_row['metrics/precision(B)'], 4),
            'Recall': round(best_row['metrics/recall(B)'], 4),
            'mAP@0.5': round(best_row['metrics/mAP50(B)'], 4),
            'mAP@0.5:0.95': round(best_row['metrics/mAP50-95(B)'], 4),
        }
    except Exception as e:
        print(f"⚠️ Error parsing results for {exp_name}: {e}")
        return None

base_model_tag = os.path.splitext(CHOSEN_MODEL_WEIGHTS)[0]
summary_data = [
    data for imgsz in RESOLUTIONS_TO_TEST
    if (data := parse_experiment_c_results(imgsz, base_model_tag)) is not None
]

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    print("\n" + "=" * 70)
    print("        EXPERIMENT C: INPUT RESOLUTION COMPARISON SUMMARY       ")
    print("=" * 70)
    print(summary_df.to_string(index=False))

    summary_csv_path = f"{EXPERIMENTS_BACKUP_PATH}/expC_resolution_comparison_summary.csv"
    summary_df.to_csv(summary_csv_path, index=False)
    print(f"\n📊 Summary table saved to: {summary_csv_path}")

### SECTION 6: Final Champion Model Export to ONNX

In [ ]:
# ==========================================
# SECTION 6: Final Champion Model Export to ONNX
# ==========================================
# Exports both resolutions to ONNX if you need to benchmark them on the Jetson
for imgsz in RESOLUTIONS_TO_TEST:
    exp_name = f"expC_{base_model_tag}_res{imgsz}"
    pt_path = f"{EXPERIMENTS_BACKUP_PATH}/{exp_name}/weights/best.pt"

    if os.path.exists(pt_path):
        print(f"\n📦 Exporting {exp_name} to ONNX (imgsz={imgsz})...")
        try:
            model = YOLO(pt_path)
            exported = model.export(format="onnx", imgsz=imgsz)
            dest = f"{ONNX_OUTPUT_DIR}/{exp_name}.onnx"
            shutil.copy(exported, dest)
            print(f"  ✅ Exported successfully: {dest}")
        except Exception as e:
            print(f"  ❌ Export failed for {exp_name}: {e}")

### SECTION 7: Runtime Disconnect (Safety)

In [ ]:
# ==========================================
# SECTION 7: Runtime Disconnect (Safety)
# ==========================================
from google.colab import runtime
runtime.unassign()